In [15]:
"""
ROC analysis for IGHV status and Migration classification systems

This script reads patient data from the provided Excel file, cleans up the
column names and performs Receiver Operating Characteristic (ROC)
analysis for two binary classification variables (IGHV status and
Migration) against continuous MRD measurements at four time points
(9 months, 12 months, 18 months and 24 months). For each combination of
classification system and endpoint, the script computes the false
positive rate (1 − specificity), the true positive rate (sensitivity)
over a range of thresholds and the area under the ROC curve (AUC).

The script saves an individual ROC plot for each combination to the
current directory and prints a summary of AUC values and the number of
observations used in the analysis.

Requirements: Python 3.8.10, pandas and scikit‑learn must be installed.
"""

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import numpy as np
import os


def encode_labels(series: pd.Series, positive_label: str) -> pd.Series:
    """Encode a categorical series into binary labels.

    Parameters
    ----------
    series : pd.Series
        The categorical data to encode.
    positive_label : str
        The value in `series` that should be considered the positive class
        (encoded as 1). All other non‑missing values are encoded as 0.

    Returns
    -------
    pd.Series
        A series of 0/1 values with NaNs preserved where the input
        contained NaNs.
    """
    return series.apply(lambda x: 1 if x == positive_label else (0 if pd.notna(x) else np.nan))


def perform_roc_analysis(df: pd.DataFrame, class_column: str, positive_label: str,
                         time_columns: list, prefix: str, out_dir: str) -> pd.DataFrame:
    """Perform ROC analysis for a specific classification column across several time points.

    Parameters
    ----------
    df : pd.DataFrame
        The data frame containing classification and MRD percentage columns.
    class_column : str
        Name of the column in `df` representing the classification variable.
    positive_label : str
        The category in `class_column` to treat as the positive class.
    time_columns : list
        A list of column names corresponding to continuous MRD measurements at
        different time points (e.g., ['9 months', '12 months']).
    prefix : str
        A prefix used when naming output figure files.
    out_dir : str
        Directory where output figures should be saved.

    Returns
    -------
    pd.DataFrame
        A summary data frame containing the AUC, number of samples and time
        point for each ROC curve generated.
    """
    summary_records = []
    # Encode the classification variable into binary labels
    labels = encode_labels(df[class_column], positive_label)

    for time_col in time_columns:
        # Select rows where both the class label and the continuous predictor are not missing
        mask = labels.notna() & df[time_col].notna()
        if mask.sum() == 0:
            # Skip if no data available
            continue
        y_true = labels.loc[mask].astype(int)
        y_scores = df.loc[mask, time_col].astype(float)
        # Compute ROC curve and AUC
        fpr, tpr, thresholds = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)
        # Plot ROC curve
        plt.figure(figsize=(6, 6))
        plt.plot(fpr, tpr, color='blue', lw=2, label=f'AUC = {roc_auc:.3f}')
        # Plot the diagonal (chance line)
        plt.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--')
        plt.xlabel('1 − Specificity (False Positive Rate)')
        plt.ylabel('Sensitivity (True Positive Rate)')
        plt.title(f'ROC Curve for {prefix} – {time_col}')
        plt.legend(loc='lower right')
        plt.grid(True, linestyle=':', linewidth=0.5)
        # Save figure
        fname = f'ROC_{prefix}_{time_col.replace(" ", "").replace("/", "_")}.png'
        fpath = os.path.join(out_dir, fname)
        plt.tight_layout()
        plt.savefig(fpath, dpi=300)
        plt.close()
        # Store summary
        summary_records.append({
            'Classification': prefix,
            'TimePoint': time_col,
            'PositiveClass': positive_label,
            'SamplesUsed': int(mask.sum()),
            'AUC': roc_auc
        })
    return pd.DataFrame(summary_records)


def main():
    # Path to the Excel file provided by the user
    excel_path = 'ROCanalysisdata.xlsx'
    out_dir = '.'  # Save figures in current directory
    # Load the dataset; skip the first blank row and use the second row as header
    df = pd.read_excel(excel_path, header=0)
    # Clean column names (strip whitespace)
    df.columns = [str(c).strip() for c in df.columns]
    # Define the continuous MRD measurement columns
    timepoints = ['MRD 9 months', 'MRD 12 months', 'MRD 18 months', 'MRD 24 months']
    # Perform ROC analysis for IGHV status
    summary_ighv = perform_roc_analysis(
        df=df,
        class_column='IGHV',
        positive_label='M-CLL',
        time_columns=timepoints,
        prefix='IGHV',
        out_dir=out_dir
    )
    # Perform ROC analysis for Migration
    summary_migration = perform_roc_analysis(
        df=df,
        class_column='Migration',
        positive_label='Responder',
        time_columns=timepoints,
        prefix='Migration',
        out_dir=out_dir
    )
    summary = pd.concat([summary_ighv, summary_migration], ignore_index=True)
    # Print summary table
    print('ROC Analysis Summary:')
    print(summary.to_string(index=False))
    # Optionally save summary to a CSV file
    summary.to_csv('roc_summary.csv', index=False)


if __name__ == '__main__':
    main()

ROC Analysis Summary:
Classification     TimePoint PositiveClass  SamplesUsed      AUC
          IGHV  MRD 9 months         M-CLL           15 0.777778
          IGHV MRD 12 months         M-CLL           15 0.851852
          IGHV MRD 18 months         M-CLL           15 0.833333
          IGHV MRD 24 months         M-CLL           15 0.787037
     Migration  MRD 9 months     Responder           15 0.954545
     Migration MRD 12 months     Responder           15 0.977273
     Migration MRD 18 months     Responder           15 1.000000
     Migration MRD 24 months     Responder           15 0.977273


In [4]:
"""
ROC analysis for IGHV status and Migration classification systems

This script reads patient data from the provided Excel file, cleans up the
column names and performs Receiver Operating Characteristic (ROC)
analysis for two binary classification variables (IGHV status and
Migration) against continuous MRD measurements at four time points
(9 months, 12 months, 18 months and 24 months). For each time point,
the script computes ROC curves for both classification systems and plots
them together on the same figure (Migration in blue, IGHV in orange).

For each curve it computes the false positive rate (1 - specificity),
the true positive rate (sensitivity) and the area under the ROC curve (AUC).

The script saves one ROC plot per time point to the current directory and
prints a summary of AUC values and the number of observations used in the
analysis.

Requirements: Python 3.8.10, pandas and scikit-learn must be installed.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc


def encode_labels(series: pd.Series, positive_label: str) -> pd.Series:
    """Encode a categorical series into binary labels.

    Parameters
    ----------
    series : pd.Series
        The categorical data to encode.
    positive_label : str
        The value in `series` that should be considered the positive class
        (encoded as 1). All other non-missing values are encoded as 0.

    Returns
    -------
    pd.Series
        A series of 0/1 values with NaNs preserved where the input
        contained NaNs.
    """
    return series.apply(
        lambda x: 1 if x == positive_label else (0 if pd.notna(x) else np.nan)
    )


def compute_roc_for_marker(
    df: pd.DataFrame,
    class_column: str,
    positive_label: str,
    time_col: str,
):
    """Compute ROC curve for a single classification marker at one time point.

    Returns
    -------
    fpr : np.ndarray
        False positive rate values.
    tpr : np.ndarray
        True positive rate values.
    roc_auc : float
        Area under the ROC curve.
    n_samples : int
        Number of observations used.
    """
    labels = encode_labels(df[class_column], positive_label)

    mask = labels.notna() & df[time_col].notna()
    n_samples = int(mask.sum())
    if n_samples == 0:
        return None, None, None, 0

    y_true = labels.loc[mask].astype(int)
    y_scores = df.loc[mask, time_col].astype(float)

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    return fpr, tpr, roc_auc, n_samples


def main():
    # Path to the Excel file provided by the user
    excel_path = "ROCanalysisdata.xlsx"
    out_dir = "."  # Save figures in current directory

    # Load the dataset
    df = pd.read_excel(excel_path, header=0)
    # Clean column names (strip whitespace)
    df.columns = [str(c).strip() for c in df.columns]

    # Define the continuous MRD measurement columns
    timepoints = ["MRD 9 months", "MRD 12 months", "MRD 18 months", "MRD 24 months"]

    summary_records = []

    for time_col in timepoints:
        # Compute ROC for Migration (blue)
        fpr_mig, tpr_mig, auc_mig, n_mig = compute_roc_for_marker(
            df=df,
            class_column="Migration",
            positive_label="Responder",
            time_col=time_col,
        )

        # Compute ROC for IGHV (orange)
        fpr_ighv, tpr_ighv, auc_ighv, n_ighv = compute_roc_for_marker(
            df=df,
            class_column="IGHV",
            positive_label="M-CLL",
            time_col=time_col,
        )

        # Skip plotting if neither curve has data
        if (n_mig == 0) and (n_ighv == 0):
            continue

        # Create combined ROC plot for this time point
        plt.figure(figsize=(6, 6))

        # Plot Migration in blue
        if n_mig > 0:
            plt.plot(
                fpr_mig,
                tpr_mig,
                color="blue",
                lw=2,
                label=f"Migration (AUC = {auc_mig:.3f}, n = {n_mig})",
            )
            summary_records.append(
                {
                    "Classification": "Migration",
                    "TimePoint": time_col,
                    "PositiveClass": "Responder",
                    "SamplesUsed": n_mig,
                    "AUC": auc_mig,
                }
            )

        # Plot IGHV in orange
        if n_ighv > 0:
            plt.plot(
                fpr_ighv,
                tpr_ighv,
                color="orange",
                lw=2,
                label=f"IGHV (AUC = {auc_ighv:.3f}, n = {n_ighv})",
            )
            summary_records.append(
                {
                    "Classification": "IGHV",
                    "TimePoint": time_col,
                    "PositiveClass": "M-CLL",
                    "SamplesUsed": n_ighv,
                    "AUC": auc_ighv,
                }
            )

        # Plot the diagonal (chance line)
        plt.plot([0, 1], [0, 1], color="grey", lw=1, linestyle="--")

        plt.xlabel("1 - Specificity (False Positive Rate)")
        plt.ylabel("Sensitivity (True Positive Rate)")
        plt.title(f"ROC Curves for Migration and IGHV - {time_col}")
        plt.legend(loc="lower right")
        plt.grid(True, linestyle=":", linewidth=0.5)

        # Save one figure per time point
        fname = f'ROC_Migration_vs_IGHV_{time_col.replace(" ", "").replace("/", "_")}.png'
        fpath = os.path.join(out_dir, fname)
        plt.tight_layout()
        plt.savefig(fpath, dpi=300)
        plt.close()

    # Build summary table
    if summary_records:
        summary = pd.DataFrame(summary_records)
        print("ROC Analysis Summary:")
        print(summary.to_string(index=False))
        summary.to_csv("roc_summary_combined.csv", index=False)
    else:
        print("No valid data to compute any ROC curves.")


if __name__ == "__main__":
    main()

ROC Analysis Summary:
Classification     TimePoint PositiveClass  SamplesUsed      AUC
     Migration  MRD 9 months     Responder           15 0.954545
          IGHV  MRD 9 months         M-CLL           15 0.777778
     Migration MRD 12 months     Responder           15 0.977273
          IGHV MRD 12 months         M-CLL           15 0.851852
     Migration MRD 18 months     Responder           15 1.000000
          IGHV MRD 18 months         M-CLL           15 0.833333
     Migration MRD 24 months     Responder           15 0.977273
          IGHV MRD 24 months         M-CLL           15 0.787037
